In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import time
import torch
import pandas as pd

import pygeohash as pgh


from collections import defaultdict
from torch_geometric.utils.convert import from_networkx
from torch_geometric.utils import to_undirected


%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# dataset = 'amazon-electronics'
dataset = 'brightkite'
# dataset = 'github'


df = pd.read_csv(f'../data/processed/{dataset}_all_events.csv', dtype={"uid": "Int64", "timestamp": "Int64", "other_uid": "Int64"})

In [ ]:
df[df['event_set'] == 'relational']['uid'].nunique()

In [ ]:
df[df['event_set'] == 'personal']['uid'].nunique()

In [ ]:
dfg = df.groupby(['uid', 'event_set']).agg(cnt=('event','count')).reset_index().groupby('uid').agg(cnt=('event_set','count')).reset_index()
dfg[dfg['cnt'] == 2]['uid'].nunique()

In [ ]:
dfg

In [ ]:
dfg2 = df.groupby(['uid']).agg(cnt=('event','count')).reset_index()
dfg2[(dfg2['cnt'] <=20) & (dfg2['cnt'] >= 15)]

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dfg2['cnt'], bins=50, edgecolor='black')

# plt.xscale('log')
plt.yscale('log')

plt.xlabel('Event count')
plt.ylabel('Number of Users (log scale)')
plt.title('Distribution of User Event Counts (Log Scale)')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
df_filtered = dfg2[dfg2['cnt'] > 0]

# Create log-spaced bins
min_cnt = df_filtered['cnt'].min()
max_cnt = df_filtered['cnt'].max()
bins = np.logspace(np.log10(min_cnt), np.log10(max_cnt), num=50)

# Plot histogram with log-scaled bins
plt.figure(figsize=(8, 5))
plt.hist(df_filtered['cnt'], bins=bins, edgecolor='black')

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Count (log scale)')
plt.ylabel('Number of Users (log scale)')
plt.title('Log-Log Histogram of User Event Counts')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
def hist(dataset, max_cnt=None):
    
    df = pd.read_csv(f'../data/processed/{dataset}_all_events.csv', dtype={"uid": "Int64", "timestamp": "Int64", "other_uid": "Int64"})

    print(dataset)

    dfg = df.groupby(['uid']).agg(cnt=('event','count')).reset_index()
    
    if max_cnt is not None:
        dfg2 = dfg[dfg['cnt'] <= max_cnt]
    else:
        dfg2 = dfg

    plt.figure(figsize=(6, 5))
    plt.hist(dfg2['cnt'], bins=50, edgecolor='black')

    # plt.xscale('log')
    plt.yscale('log')

    plt.xlabel('Event count')
    plt.ylabel('Number of Users (log scale)')
    plt.title(f'Distribution of User Event Counts in pres-{dataset}')
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    if max_cnt is not None:
        plt.xlim(0, max_cnt)  # Set y-axis limits as desired
    
    plt.tight_layout()
    plt.show()

In [ ]:
def stats(dataset):
    df = pd.read_csv(f'../data/processed/{dataset}_all_events.csv', dtype={"uid": "Int64", "timestamp": "Int64", "other_uid": "Int64"})

    print(dataset)

    print(f"All events: {df.shape[0]}")
    print(f"Personal events: {df[df['event_set'] == 'personal'].shape[0]}")
    print(f"Relational events: {df[df['event_set'] == 'relational'].shape[0]}")
    print(f"User: {df['uid'].nunique()}")
    print(f"Unique events: {df['event'].nunique()}")
    print(f"Unique timestamp: {df['timestamp'][df['timestamp'] > 0].nunique()}")
    print(f"Duration timestamp: {(df['timestamp'][df['timestamp'] > 0].max() - df['timestamp'][df['timestamp'] > 0].min())/86400} days")
    
    print(f"Wrong Unique timestamp: {df['timestamp'].nunique()}")
    print(f"Wrong Duration timestamp: {(df['timestamp'].max() - df['timestamp'].min())/86400} days")
    
    print(f"User with Personal events: {df[df['event_set'] == 'personal']['uid'].nunique()}")
    print(f"User with Relational events: {df[df['event_set'] == 'relational']['uid'].nunique()}")
    
    dfg = df.groupby(['uid', 'event_set']).agg(cnt=('event','count')).reset_index().groupby('uid').agg(cnt=('event_set','count')).reset_index()
    print(f"User with Both Events {dfg[dfg['cnt'] == 2]['uid'].nunique()}")

In [ ]:
stats('brightkite')

In [ ]:
stats('gowalla')

In [ ]:
stats('amazon-clothing')

In [ ]:
stats('amazon-electronics')

In [ ]:
stats('github')

In [ ]:
df = pd.read_csv(f'../data/processed/brightkite_all_events.csv', dtype={"uid": "Int64", "timestamp": "Int64", "other_uid": "Int64"})
df

In [ ]:
hist('brightkite', 2500)

In [ ]:
hist('gowalla', 2500)

In [ ]:
hist('amazon-clothing', 200)

In [ ]:
hist('amazon-electronics', 1000)

In [ ]:
hist('github', 10000)